# NB_Dynamic_Alerts_BronzeToSilver
**Source:** `AI_Medallion.Bronze.dynamic_alerts`
**Target:** `AI_Medallion.Silver.dynamic_alerts`
**Grain:** One row per `dynamic_alerts_notification_id`
**Load type:** Overwrite
**Rule:** Standardize column names (prefix `dynamic_alerts_`, snake_case), TRIM all text, cast publication date to `date`.

In [1]:
# Workspace folder: C:\\spark-dev\\CoE_transformation_framework (see README)
import os
from pathlib import Path

os.environ["HADOOP_HOME"] = r"C:\hadoop\hadoop-3.5.0"
os.environ["HADOOP_PREFIX"] = os.environ["HADOOP_HOME"]
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]
 
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])
print("HADOOP_PREFIX:", os.environ["HADOOP_PREFIX"])
print("winutils exists:", os.path.isfile(
    os.path.join(os.environ["HADOOP_HOME"], "bin", "winutils.exe")
))
_cwd = Path.cwd()
if not (_cwd / "Common" / "bootstrap.py").is_file():
    _repo = _cwd.parent
    if (_repo / "Common" / "bootstrap.py").is_file():
        os.chdir(_repo)
REBUILD_MEDALLION_CACHE = True  # True: rebuild; False: use cache
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
from Common.io import sync_rebuild_medallion_cache_env

sync_rebuild_medallion_cache_env(REBUILD_MEDALLION_CACHE)


HADOOP_HOME: C:\hadoop\hadoop-3.5.0
HADOOP_PREFIX: C:\hadoop\hadoop-3.5.0
winutils exists: True


In [2]:
import sys
print(sys.executable)

c:\spark-dev\CoE_transformation_framework\.venv\Scripts\python.exe


In [3]:
%run ./Common/bootstrap

Bootstrap OK: runtime=local mode=created spark=3.5.4 pyspark=3.5.4 path_reads=True onelake_read=deltalake-scan
Repo root: C:\spark-dev\CoE_Transformation_Framework_copilot
spark.jars.packages=io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-azure:3.3.4,com.azure:azure-storage-blob:12.25.1
OneLake reads: deltalake scan() + az login -> Spark (column mapping; avoids local PySpark abfss hang).
TABLE_CURSOR_BRONZE -> abfss://4cecee03-0629-4081-8ae8-b749ee04d43f@onelake.dfs.fabric.microsoft.com/1ee46463-2495-44f5-9dc9-d635d0067483/Tables/Bronze/cursor_usage
Tip: set MEDALLION_DEBUG_READ=1 before bootstrap to log read progress.
Tip: large tables — read_table() one at a time; transient Azure errors auto-retry.
Local table cache ON (default): parquet under ~/.fabric/medallion_cache/. Set REBUILD_MEDALLION_CACHE = True in settings to force OneLake. reload_io_helpers(globals()) after git pull — no kernel restart.
OneLake writes: deltalake + az login (merge/overwrite). Set MEDALLION_ONELAK

In [4]:
# TODO: move to Common/config.py if not defined there yet
TABLE_DYNAMIC_ALERTS_BRONZE = globals().get(
    "TABLE_DYNAMIC_ALERTS_BRONZE", "AI_Medallion.Bronze.dynamic_alerts"
)
TABLE_DYNAMIC_ALERTS_SILVER = globals().get(
    "TABLE_DYNAMIC_ALERTS_SILVER", "AI_Medallion.Silver.dynamic_alerts"
)

bronze = read_table_broken_lineage(TABLE_DYNAMIC_ALERTS_BRONZE, "dynamic_alerts_bronze")
bronze.printSchema()

read_table_broken_lineage: scan -> local cache (AI_Medallion.Bronze.dynamic_alerts)
read_table: deltalake scan abfss://4cecee03-0629-4081-8ae8-b749ee04d43f@onelake.dfs.fabric.microsoft.com/1e… (pass 1/6)
read_table: OneLake object-store concurrency limit=1 (raise MEDALLION_OBJECT_STORE_CONCURRENCY if reads are too slow)
read_table: deltalake scan() streaming batches…
read_table: scan streaming… 998 rows from OneLake so far
read_table: scan finished (998 rows)
read_table: opening local parquet cache (998 rows) -> C:\Users\soukelkr\.fabric\medallion_cache\AI_Medallion__Bronze__dynamic_alerts
read_table_broken_lineage: opened cache after scan (dynamic_alerts_bronze)
root
 |-- NotificationID: string (nullable = true)
 |-- NotificationType: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- IsActive: string (nullable = true)
 |-- PublicationDate: string (nullable = true)
 |-- Severity: string (nullable = true)
 |-- execution_date: string (nullable = true)



In [5]:
from Common.transforms import _bronze_column
 
def _txt(*names):
    """TRIM(column) as string; accepts PascalCase or snake_case bronze names."""
    return F.trim(_bronze_column(bronze, *names).cast("string"))


_pub = _txt("PublicationDate", "publication_date")
_pub_date = F.coalesce(
    F.to_date(_pub),
    F.to_date(_pub, "dd/MM/yyyy"),
    F.to_date(_pub, "MM/dd/yyyy"),
    F.to_date(F.to_timestamp(_pub)),
)

_execution_date = _bronze_column(bronze, "execution_date").cast("timestamp")
 

silver = (
    bronze.select(
        _txt("NotificationID", "notification_id").alias("dynamic_alerts_notification_id"),
        _txt("NotificationType", "notification_type").alias("dynamic_alerts_notification_type"),
        _txt("Description", "description").alias("dynamic_alerts_description"),
        _txt("IsActive", "is_active").alias("dynamic_alerts_is_active"),
        _pub_date.alias("dynamic_alerts_publication_date"),
        _txt("Severity", "severity").alias("dynamic_alerts_severity"),
        _execution_date.alias("dynamic_alerts_execution_date"),
     )
    .filter(
        F.col("dynamic_alerts_notification_id").isNotNull()
        & (F.col("dynamic_alerts_notification_id") != "")
    )
    .dropDuplicates(["dynamic_alerts_notification_id"])
)

silver.printSchema()
silver.orderBy("dynamic_alerts_notification_id").show(truncate=False)
print(
    "rows:", silver.count(),
    "| distinct ids:", silver.select("dynamic_alerts_notification_id").distinct().count(),
)

root
 |-- dynamic_alerts_notification_id: string (nullable = true)
 |-- dynamic_alerts_notification_type: string (nullable = true)
 |-- dynamic_alerts_description: string (nullable = true)
 |-- dynamic_alerts_is_active: string (nullable = true)
 |-- dynamic_alerts_publication_date: date (nullable = true)
 |-- dynamic_alerts_severity: string (nullable = true)
 |-- dynamic_alerts_execution_date: timestamp (nullable = true)

+------------------------------+--------------------------------+--------------------------------------------------------------------------------+------------------------+-------------------------------+-----------------------+-----------------------------+
|dynamic_alerts_notification_id|dynamic_alerts_notification_type|dynamic_alerts_description                                                      |dynamic_alerts_is_active|dynamic_alerts_publication_date|dynamic_alerts_severity|dynamic_alerts_execution_date|
+------------------------------+--------------------------

In [6]:
# Date parse check: rows where bronze had a value but the cast failed
bad_dates = silver.filter(
    F.col("dynamic_alerts_publication_date").isNull()
).count()
print("publication_date null after cast:", bad_dates)

publication_date null after cast: 0


In [7]:
apply_local_merge_spark_conf(spark)
 
write_full_table(silver, TABLE_DYNAMIC_ALERTS_SILVER)
 
show_sample(silver.limit(10))

publish_merge_staging: AI_Medallion.Silver.dynamic_alerts -> C:\Users\soukelkr\.fabric\merge_staging\AI_Medallion__Silver__dynamic_alerts


materialize staging: Spark JVM parquet (8 partitions)
deltalake write mode=overwrite abfss://4cecee03-0629-4081-8ae8-b749ee04d43f@onelake.dfs.fabric.microsoft.com/1ee46463-2495-44f5-9dc9-d635d0067483/Tables/Silver/dynamic_alerts (local staging -> OneLake)...
deltalake write chunk 1 (1 rows) mode=overwrite
deltalake write chunk 2 (2 rows) mode=append
deltalake write done


DataFrame[dynamic_alerts_notification_id: string, dynamic_alerts_notification_type: string, dynamic_alerts_description: string, dynamic_alerts_is_active: string, dynamic_alerts_publication_date: date, dynamic_alerts_severity: string, dynamic_alerts_execution_date: timestamp]